**HAVYN Stage 5 - Standard Model Training**

Train the Standard HAVYN model (Qwen3-8B, 4-bit LoRA) from the frozen `havyn-train-v1` dataset on a Colab T4.

`havyn-stage5/scripts/train_lora.py` is the single source of truth. This notebook verifies the environment, the repository and the frozen dataset, runs a construction preflight, and launches that script. Only the training cell starts training.

T4 hardware plan: 4-bit NF4 weights, sequence length 2048, per-device batch 1, gradient accumulation 16 (effective batch 16), gradient checkpointing, FP16 compute on Turing (BF16 only on Ampere or newer).

Run the cells in order. Every setup cell is safe to re-run.


In [ ]:
import os
import platform
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv"],
    check=False,
)

import torch

print("torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", capability)
    print(f"VRAM total: {total / 1024**3:.2f} GB")
    print(f"VRAM free: {free / 1024**3:.2f} GB")
    print("BF16 reported:", torch.cuda.is_bf16_supported())
    print("Precision selected:", "bf16" if capability[0] >= 8 and torch.cuda.is_bf16_supported() else "fp16")
else:
    raise SystemExit("No CUDA GPU. Select Runtime > Change runtime type > T4 GPU.")


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/havyn-stage5")
DATASET_DIR = DRIVE / "datasets" / "havyn-train-v1"
CHECKPOINT_DIR = DRIVE / "checkpoints" / "standard"
REPORT_DIR = DRIVE / "reports" / "training-standard"

for path in (DRIVE, DATASET_DIR, CHECKPOINT_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE)
print("Dataset:", DATASET_DIR)
print("Checkpoints:", CHECKPOINT_DIR)
print("Reports:", REPORT_DIR)


In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/Havyn")
BRANCH = "stage5-colab"


def run(*args, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise SystemExit("command failed: " + " ".join(args))
    return result.stdout.strip()


if not (REPO / ".git").exists():
    print(run("git", "clone", "https://github.com/HamzaNinjaCoder1/Havyn.git", str(REPO)))
else:
    print(run("git", "fetch", "origin", cwd=REPO))

if run("git", "rev-parse", "--is-inside-work-tree", cwd=REPO) != "true":
    raise SystemExit(str(REPO) + " is not a git work tree")

dirty = run("git", "status", "--porcelain", cwd=REPO)
if dirty:
    print("Discarding local Colab edits to match origin:")
    print(dirty)

run("git", "checkout", "-B", BRANCH, "origin/" + BRANCH, cwd=REPO)

head = run("git", "rev-parse", "HEAD", cwd=REPO)
remote = run("git", "rev-parse", "origin/" + BRANCH, cwd=REPO)
if head != remote:
    raise SystemExit("HEAD does not match origin/" + BRANCH)

print("Branch:", run("git", "branch", "--show-current", cwd=REPO))
print("HEAD:", head)
print(run("git", "log", "--oneline", "-5", cwd=REPO))


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/Havyn")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "havyn-stage5/requirements-colab.txt"],
    cwd=REPO,
    check=True,
)

import accelerate
import bitsandbytes
import datasets
import peft
import transformers
import trl

print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("datasets:", datasets.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("accelerate:", accelerate.__version__)

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata

        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

print("HF_TOKEN available:", bool(hf_token))
if hf_token:
    from huggingface_hub import login

    login(token=hf_token, add_to_git_credential=False)


In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

REPO = Path("/content/Havyn")
SRC = REPO / "havyn-stage5" / "datasets" / "havyn-train-v1"
DST = Path("/content/drive/MyDrive/havyn-stage5/datasets/havyn-train-v1")
FILES = ("train.jsonl", "validation.jsonl", "heldout.jsonl", "audit-report.json", "dataset-card.json")


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = json.loads((SRC / "FROZEN.json").read_text(encoding="utf-8"))
for name, expected in manifest["sha256"].items():
    actual = sha256(SRC / name)
    if actual != expected:
        raise SystemExit("repo dataset does not match FROZEN.json: " + name)

DST.mkdir(parents=True, exist_ok=True)
for name in (*FILES, "FROZEN.json"):
    source = SRC / name
    target = DST / name
    if not target.exists() or sha256(target) != sha256(source):
        shutil.copy2(source, target)
        print("synced:", name)

drive_manifest = json.loads((DST / "FROZEN.json").read_text(encoding="utf-8"))
for name, expected in drive_manifest["sha256"].items():
    actual = sha256(DST / name)
    if actual != expected:
        raise SystemExit("Drive dataset does not match FROZEN.json: " + name)

counts = {}
for name in ("train.jsonl", "validation.jsonl", "heldout.jsonl"):
    with (DST / name).open(encoding="utf-8") as handle:
        counts[name] = sum(1 for line in handle if line.strip())

print("frozen_at:", drive_manifest["frozen_at"])
print("rows:", counts)
print("dataset check: PASS")


In [ ]:
import json
import sys
from pathlib import Path

import torch
import yaml

REPO = Path("/content/Havyn")
CONFIG = REPO / "havyn-stage5" / "configs" / "train-standard.yaml"
DATASET_DIR = Path("/content/drive/MyDrive/havyn-stage5/datasets/havyn-train-v1")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/havyn-stage5/checkpoints/standard")

sys.path.insert(0, str(REPO / "havyn-stage5" / "scripts"))
from train_lora import latest_valid_checkpoint

cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
train_cfg = cfg["train"]
batch = int(train_cfg["per_device_train_batch_size"])
accumulation = int(train_cfg["gradient_accumulation_steps"])
resume_target = latest_valid_checkpoint(CHECKPOINT_DIR)
free, total = torch.cuda.mem_get_info()
capability = torch.cuda.get_device_capability(0)
precision = "bf16" if capability[0] >= 8 and torch.cuda.is_bf16_supported() else "fp16"

counts = {}
for name in ("train.jsonl", "validation.jsonl"):
    with (DATASET_DIR / name).open(encoding="utf-8") as handle:
        counts[name] = sum(1 for line in handle if line.strip())

print("=" * 40)
print("HAVYN STAGE 5 STANDARD TRAINING")
print("=" * 40)
print("Base model:", cfg["base_model"])
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {total / 1024**3:.2f} GB total / {free / 1024**3:.2f} GB free")
print("Precision:", precision)
print("Sequence length:", cfg["max_seq_length"])
print("Per-device batch:", batch)
print("Gradient accumulation:", accumulation)
print("Effective batch:", batch * accumulation)
print("Epochs:", train_cfg["epochs"])
print("Learning rate:", train_cfg["learning_rate"])
print("Warmup ratio:", train_cfg["warmup_ratio"])
print("Train rows:", counts["train.jsonl"])
print("Validation rows:", counts["validation.jsonl"])
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Checkpoint found:", resume_target is not None)
print("Training will:", "resume from " + str(resume_target) if resume_target else "start fresh")
print("=" * 40)


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/Havyn")

command = [
    sys.executable,
    "havyn-stage5/scripts/train_lora.py",
    "--config", "havyn-stage5/configs/train-standard.yaml",
    "--dataset-dir", "/content/drive/MyDrive/havyn-stage5/datasets/havyn-train-v1",
    "--output-dir", "/content/drive/MyDrive/havyn-stage5/checkpoints/standard",
    "--preflight",
]
print(" ".join(command), flush=True)
result = subprocess.run(command, cwd=REPO, text=True)
if result.returncode != 0:
    raise SystemExit("preflight failed with exit code " + str(result.returncode))
print("preflight: PASS")


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/Havyn")

command = [
    sys.executable,
    "havyn-stage5/scripts/train_lora.py",
    "--config", "havyn-stage5/configs/train-standard.yaml",
    "--dataset-dir", "/content/drive/MyDrive/havyn-stage5/datasets/havyn-train-v1",
    "--output-dir", "/content/drive/MyDrive/havyn-stage5/checkpoints/standard",
    "--resume",
]
print(" ".join(command), flush=True)
result = subprocess.run(command, cwd=REPO, text=True)
print("training exit code:", result.returncode)
if result.returncode != 0:
    raise SystemExit("training process failed with exit code " + str(result.returncode))


In [ ]:
import json
import sys
from pathlib import Path

REPO = Path("/content/Havyn")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/havyn-stage5/checkpoints/standard")
REPORT = Path("/content/drive/MyDrive/havyn-stage5/reports/training-standard/training-report.json")

sys.path.insert(0, str(REPO / "havyn-stage5" / "scripts"))
from train_lora import latest_valid_checkpoint

checkpoints = sorted(
    (path for path in CHECKPOINT_DIR.glob("checkpoint-*") if path.is_dir()),
    key=lambda path: path.stat().st_mtime,
)

print("checkpoint count:", len(checkpoints))
for path in checkpoints:
    size = sum(item.stat().st_size for item in path.rglob("*") if item.is_file())
    print(f"  {path.name}: {size / 1024**2:.1f} MB")

print("latest checkpoint:", latest_valid_checkpoint(CHECKPOINT_DIR))
print("adapter dir exists:", (CHECKPOINT_DIR / "adapter").is_dir())
print("report exists:", REPORT.exists())

if REPORT.exists():
    report = json.loads(REPORT.read_text(encoding="utf-8"))
    print("reported status:", report.get("status"))
    print("started:", report.get("started"))
    print("finished:", report.get("finished"))
    print("final loss:", report.get("final_loss"))


In [ ]:
import json
from pathlib import Path

CHECKPOINT_DIR = Path("/content/drive/MyDrive/havyn-stage5/checkpoints/standard")
REPORT = Path("/content/drive/MyDrive/havyn-stage5/reports/training-standard/training-report.json")

adapter = CHECKPOINT_DIR / "adapter"
adapter_files = sorted(path.name for path in adapter.glob("*")) if adapter.is_dir() else []
report = json.loads(REPORT.read_text(encoding="utf-8")) if REPORT.exists() else {}
checkpoints = sorted(
    (path for path in CHECKPOINT_DIR.glob("checkpoint-*") if path.is_dir()),
    key=lambda path: path.stat().st_mtime,
)

if report.get("status") == "blocked":
    state = "BLOCKED"
elif report.get("status") == "failed":
    state = "FAILED"
elif report.get("status") == "completed":
    state = "COMPLETED" if adapter_files else "FAILED"
elif checkpoints:
    state = "STARTED"
elif report:
    state = "FAILED"
else:
    state = "NOT STARTED"

print("=" * 40)
print("HAVYN STAGE 5 TRAINING EVIDENCE")
print("=" * 40)
print("State:", state)
print("Report status:", report.get("status"))
print("Checkpoints:", [path.name for path in checkpoints])
print("Adapter files:", adapter_files)
print("Final loss:", report.get("final_loss"))
print("Resume target:", report.get("resumed_from"))
print("=" * 40)

if state == "COMPLETED":
    print("Training completed. Proof: trainer report status and adapter files both present.")
elif state == "STARTED":
    print("Next action: run the training cell; it resumes from the latest checkpoint.")
elif state == "FAILED" and checkpoints:
    print("Next action: run the training cell; it resumes from the latest checkpoint.")
elif state == "FAILED":
    print("Next action: run the preflight cell and read the error before changing any value.")
else:
    print("Next action: run the preflight cell, then the training cell.")
